# RAG Pipeline Exercise

In this exercise you will build and **compare two simple Retrieval-Augmented Generation (RAG) pipelines**.

You will work with a small collection of PDF documents (e.g. medical guidelines) and:

1. Load and chunk the PDF documents.
2. Create a vector index using **embedding model A** (local `BAAI/bge-m3`).
3. Create a second index using **embedding model B** (e.g. OpenAI or Gemini embeddings).
4. Implement a simple **retriever** and an **answering function** that calls an LLM with retrieved context.
5. Automatically **generate questions** from the documents and use them to **compare two RAG configurations**.

Cells marked with `# TODO` are **for students to implement**.
Everything else is provided scaffolding.

## 0. Setup & Imports

In [1]:
# TODO (easy): skim the imports and make sure you understand what each library is used for.

from dotenv import load_dotenv
import os
import glob
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from sentence_transformers import SentenceTransformer
import pickle
import random
import numpy as np

# LLM / API clients (we will mainly use OpenAI here; Gemini can be added as a bonus)
from openai import OpenAI

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load API keys from .env (you need to create this file once and add your keys)
load_dotenv()

deepinfra_key = os.getenv("DEEPINFRA_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

## 1. Load PDF documents

We assume there is a `data/` folder containing one or more PDF files.

**Task:** implement `load_pdfs(glob_path)` so that it:
- Iterates over all PDF files matching `glob_path`
- Reads them with `PdfReader`
- Concatenates the text of all pages into **one long string**.

In [3]:
def load_pdfs(glob_path: str = "data/*.pdf") -> str:
    """Load all PDFs matching the pattern and return their combined text.

    TODO:
    - Use `glob.glob(glob_path)` to iterate over file paths
    - For each file, open it in binary mode and create a `PdfReader`
    - Loop over `reader.pages` and extract text with the extract_text() function
    - Concatenate everything into a single string `text`
    - Be robust: skip pages where `extract_text()` returns None
    """
    # YOUR CODE HERE
    text = ""
    for pdf_path in glob.glob(glob_path):
        with open(pdf_path, "rb") as file:
            reader = PdfReader(file)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text is not None:
                    text += page_text

        
    return text


In [4]:
# Run once and inspect
raw_text = load_pdfs("data/*.pdf")
print("Number of characters:", len(raw_text))
print("Preview:", raw_text[:500])

Number of characters: 230592
Preview: Asthma: diagnosis, 
moni toring and chr onic 
asthma manag emen t (BTS, 
NICE, SI GN) 
NICE guideline 
Published: 27 No vember 202 4 
www .nice.or g.uk/guidance/ng2 45 
© NICE 202 4. All right s reserved. Subject t o Notice of right s (https://www .nice.or g.uk/t erms-and-
conditions#notice-of -right s).Your r esponsi bility 
The r ecommendations in t his guideline r epresent t he view of NICE, arriv ed at aft er car eful 
consideration of t he evidence a vailable. When e xercising t heir judgem


## 2. Chunk the text

We will split the long text into overlapping chunks.

Later you can **experiment** with different `chunk_size` and `chunk_overlap` to see how it affects retrieval.

**Task:** start with the given parameters, run once, then try at least one alternative configuration and note the effects.

In [5]:
# Base Configuration (RAG A)
chunk_size_a = 2000
chunk_overlap_a = 200

splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_a,
    chunk_overlap=chunk_overlap_a
)

# Split the raw_text into chunks for RAG A
chunks_a = splitter_a.split_text(raw_text)
print(f"RAG A: {len(chunks_a)} chunks produced, first chunk length = {len(chunks_a[0])}")

# TODO (mini-experiment): change chunk_size / chunk_overlap for RAG B
# Define chunk size and overlap for RAG B
chunk_size_b = 100  # Smaller chunk size
chunk_overlap_b = 20  # Overlap between chunks

# Initialize the splitter for RAG B
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size_b,
    chunk_overlap=chunk_overlap_b
)

# Split the raw_text into chunks for RAG B
chunks_b = splitter_b.split_text(raw_text)

# Print the results for RAG B
print(f"RAG B: {len(chunks_b)} chunks produced, first chunk length = {len(chunks_b[0])}")

RAG A: 129 chunks produced, first chunk length = 1994
RAG B: 3146 chunks produced, first chunk length = 85


## 3. Create embeddings and a FAISS index

We start with **Embedding model A: `BAAI/bge-small-en`** using `sentence-transformers`. You can find a list of more models here: https://huggingface.co/spaces/mteb/leaderboard 
make sure that the models are not bigger than the one used here. Otherwise the embeddings process will take too long.

Then, as an optional extension, you can build **Embedding model B** using OpenAI or Gemini and compare.

To keep the exercise manageable, the base version only **requires** BGE.

In [6]:
# Embedding model A (local)
model_name_a = "BAAI/bge-small-en"
embedder_a = SentenceTransformer(model_name_a)

# Compute embeddings for all chunks of configuration A
embeddings_a = embedder_a.encode(chunks_a, convert_to_numpy=True)

dimensions_a = embeddings_a.shape[1]
print("Embedding dimensionality (A):", dimensions_a)

index_a = faiss.IndexFlatL2(dimensions_a)
index_a.add(embeddings_a)
print("FAISS index (A) size:", index_a.ntotal)

# Persist index/chunks if you like (optional)
os.makedirs("faiss", exist_ok=True)
faiss.write_index(index_a, "faiss/faiss_index_a.index")
with open("faiss/chunks_a.pkl", "wb") as f:
    pickle.dump(chunks_a, f)

Embedding dimensionality (A): 384
FAISS index (A) size: 129


In [9]:
# Initialize OpenAI client
openai_client = OpenAI(api_key=openai_api_key)

# Compute embeddings for chunks_b in batches to avoid invalid '$.input' errors for large requests.
# Ensure every chunk is a plain string and handle empty lists.
if not chunks_b:
    embeddings_b = np.zeros((0, 1), dtype=np.float32)
    index_b = faiss.IndexFlatL2(1)
    print("No chunks to embed. Created empty FAISS index (B).")
else:
    batch_size = 64  # safe batch size; lower if you still hit request size issues
    all_embeddings = []
    for i in range(0, len(chunks_b), batch_size):
        batch = chunks_b[i : i + batch_size]
        # Ensure values are strings
        batch = [str(x) for x in batch]
        try:
            resp = openai_client.embeddings.create(
                model="text-embedding-3-small",
                input=batch
            )
        except Exception as e:
            # surface the error for debugging
            raise RuntimeError(f"Embedding request failed for batch starting at {i}: {e}")

        # Extract embeddings from response
        for item in resp.data:
            all_embeddings.append(item.embedding)

    # Convert to NumPy array
    embeddings_b = np.array(all_embeddings, dtype=np.float32)

    # Check the dimensionality of the embeddings
    dim_b = embeddings_b.shape[1]  # size of embedding vector

    # Create a new FAISS index
    index_b = faiss.IndexFlatL2(dim_b)  # L2 distance index

    # Add embeddings to the FAISS index
    index_b.add(embeddings_b)

    # Print the FAISS index size
    print("FAISS index (B) size:", index_b.ntotal)

FAISS index (B) size: 3146


## 4. Implement a simple retriever

We now implement a generic retrieval function that:
1. Embeds the query.
2. Searches the FAISS index.
3. Returns the corresponding text chunks.

We implement it for configuration A. If you built configuration B, you can reuse the same function.

In [13]:
def retrieve_texts(query: str, k: int, index, chunks, embedder) -> list:
    """Return the top-k most similar chunks for a query.
    - Encode the query with `embedder.encode(...)` (SentenceTransformer)
      or call the OpenAI client embeddings API when `embedder` is the OpenAI client.
    - Call `index.search(query_embedding, k)`
    - Use the returned indices to select the chunks
    - Return a list of strings (chunks)
    """
    # Create the query embedding depending on the embedder type
    if hasattr(embedder, "encode"):
        # e.g. SentenceTransformer
        query_emb = embedder.encode([query], convert_to_numpy=True)
    else:
        # assume OpenAI client (has embeddings.create)
        resp = embedder.embeddings.create(model="text-embedding-3-small", input=query)
        emb = resp.data[0].embedding
        query_emb = np.array([emb], dtype=np.float32)

    # Search FAISS index (ensure correct shape)
    distances, indices = index.search(query_emb, k)

    # Build retrieved list, guard against invalid indices (e.g. -1)
    retrieved = []
    for i in indices[0]:
        if i == -1:
            continue
        retrieved.append(chunks[i])
    return retrieved

# Quick sanity check
test_query = "What is the most important factor in diagnosing asthma?"
retrieved_test = retrieve_texts(test_query, k=5, index=index_b, chunks=chunks_b, embedder=openai_client)
print("Number of retrieved chunks:", len(retrieved_test))
print("Preview of first chunk:", retrieved_test[0] if retrieved_test else "No chunks retrieved")

Number of retrieved chunks: 5
Preview of first chunk: asthma 
• evidence r eview  B: diagnostic t est accuracy f or br onchodilat or reversibility in


## 5. Implement `answer_query` using an LLM

Now we build the actual RAG call:

1. Use `retrieve_texts` to get top-`k` chunks.
2. Concatenate them into a context string.
3. Build a prompt that:
   - shows the context
   - asks the model to answer the user question based **only** on this context.
4. Call the OpenAI chat completion API.

This is the **core RAG function**.

In [15]:
def answer_query(query: str, k: int, index, chunks, embedder, client: OpenAI) -> str:
    """RAG-style answer: retrieve context and ask an LLM."""
    # Retrieve relevant chunks
    retrieved_chunks = retrieve_texts(query, k, index, chunks, embedder)
    
    # Join the chunks into a single context string
    context = "\n".join(retrieved_chunks)

    # Create a system prompt with the context
    system_prompt = (
        f"You are a helpful assistant. Answer the following question based on the context provided:\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}"
    )

    # Prepare the message for the model
    messages = [
        {"role": "system", "content": system_prompt},
    ]

    # Call the OpenAI model to generate a completion
    # Use an available model (replace 'gpt-5o-mini' which may not exist for your account)
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # Return the model's answer text
    return completion.choices[0].message.content.strip()

# Quick manual test
answer = answer_query(
    test_query, 
    k=3, 
    index=index_b,  # Assuming you intend to use the index for RAG B
    chunks=chunks_b,  # Chunks for RAG B
    embedder=openai_client,  # Ensure this is the correct embedder
    client=openai_client
)
print("RAG answer:", answer)


RAG answer: The most important factor in diagnosing asthma is the confirmation of asthma through specific diagnostic tests, such as FeNO (fractional exhaled nitric oxide), bronchodilator reversibility (BDR), or peak expiratory flow (PEF) variability. If these tests do not confirm asthma but the clinical suspicion remains, further evaluation may be necessary to reach a definitive diagnosis.


## 6. Generate questions from random chunks (automatic evaluation set)

To compare two RAG configurations, we need **questions**.

We will:
- randomly sample a few chunks from the corpus,
- ask an LLM to generate a **good question** whose answer is contained in the chunk.

Then we can use these question–chunk pairs as a small evaluation set.

We provide most of the implementation. Your job is mainly to:
- inspect the code,
- understand the prompt,
- maybe tweak the number of chunks or retries.

In [16]:
def generate_questions_for_random_chunks(chunks, num_chunks: int = 5, max_retries: int = 2):
    selected_chunks = random.sample(chunks, num_chunks)
    qa_pairs = []

    for chunk in selected_chunks:
        prompt = prompt = (
            "Based on the following text, generate an insightful question that covers its key content:\n\n"
            "Text:\n" + chunk + "\n\n"
            "Question:"
        )

        question = None
        for attempt in range(max_retries):
            try:
                completion = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}]
                )
                question = completion.choices[0].message.content.strip()
                if question:
                    break
            except Exception as e:
                print("Error while generating question, retrying...", e)

        if question is None:
            question = "Error: could not generate question."

        qa_pairs.append((chunk, question))

    return qa_pairs

questions = generate_questions_for_random_chunks(chunks_a, num_chunks=5, max_retries=2)
for i, (chunk, q) in enumerate(questions, 1):
    print(f"Q{i}: {q}\n  From chunk preview: {chunk[:120]}...\n")

Q1: What are the recommended guidelines for assessing and ensuring proper use of inhalers in asthma management, particularly regarding patient consultations and device selection?
  From chunk preview: include t he medicines t hey contain, ho w they work, when t hey should be tak en 
and t he corr ect t echnique t o use ...

Q2: How have the NICE guidelines on hypertension in adults evolved over time, particularly in relation to recommendations concerning diagnosis and management for individuals with specific conditions such as type 2 diabetes and the elderly?
  From chunk preview: groups 
• the inf ormation on when t o refer to the hyper tension in pr egnancy guideline was made 
clear er. Hyper tens...

Q3: What recommendations does NICE provide for monitoring and managing hypertension, particularly in relation to the use of clinic measurements, home blood pressure monitoring (HBPM), and addressing issues such as postural hypotension?
  From chunk preview: © NICE 202 4. All right s res

## 7. Compare two RAG configurations

Now we can:
- Use the generated questions,
- Answer them with RAG configuration A (BGE + chunking A),
- (Optional) Answer them with RAG configuration B (e.g. different chunking and/or different embeddings),
- Compare the answers qualitatively.

To keep the exercise manageable, we start with config A only.
If you implemented config B, reuse `answer_query` with `index_b`, `chunks_b`, and your second embedder.

In [17]:
def answer_generated_questions(question_tuples, k, index, chunks, embedder, client):
    results = []
    for chunk, question in question_tuples:
        answer = answer_query(question, k, index, chunks, embedder, client)
        results.append({
            "chunk": chunk,
            "question": question,
            "answer": answer
        })
    return results

results_a = answer_generated_questions(
    questions,
    k=5,
    index=index_a,
    chunks=chunks_a,
    embedder=embedder_a,
    client=openai_client,
)

for item in results_a:
    print("Question:", item["question"])
    print("Answer A:", item["answer"])
    print("Source chunk preview:", item["chunk"][:150], "...")
    print("-" * 60)

Question: What are the recommended guidelines for assessing and ensuring proper use of inhalers in asthma management, particularly regarding patient consultations and device selection?
Answer A: The recommended guidelines for assessing and ensuring proper use of inhalers in asthma management emphasize several key areas, particularly during patient consultations and device selection. Here are the essential points based on the provided context and general best practices:

1. **Understanding Patient Needs**: During consultations, healthcare professionals should consider the individual needs of the patient when selecting inhaler devices. This includes discussing the patient's experience with previous inhalers, their ability to use different types of devices, and any preferences they might have.

2. **Inhaler Technique Assessment**: It is crucial to evaluate the patient’s inhaler technique. Healthcare providers should check if patients can correctly use their inhalers and provide guidance o

Add RAG B and create a comparison table

If you implemented a second configuration (e.g. different chunking + OpenAI embeddings):

1. Build `index_b` and `embedder_b`.
2. Run `results_b = answer_generated_questions(..., index_b, chunks_b, embedder_b, client)`.
3. For each question, compare:
   - Which answer is more complete / specific?
   - Which one is better grounded in the source chunk?
4. Summarise your findings in a short **markdown cell** or a small table.

---

This concludes the core RAG exercise.